# 04 - Ingestão Bronze: Micro-Lotes em Delta Lake
**Squad 1 — Data Quality em Tempo Real | Dupla 1**
**Integrantes:** Gabriel Franz Simoni & Carlos Eduardo Santos de Souza
**Tabelas:** `ecommerce_itens_pedido` e `ecommerce_rastreamento_entregas`

### Regras/KPIs gerados por este notebook:
Nenhuma regra de qualidade (a Bronze é cópia fiel do dado bruto, sem validação — isso só acontece no `05_silver_ingestao_qualidade.ipynb`). Este notebook gera apenas metadados de auditoria/controle:
- `bronze_ingested_at`, `ano`, `mes`, `dia`, `hora` (metadados de ingestão, usados para particionamento)
- `bronze_source_file` (rastreabilidade: de qual arquivo da camada `raw` cada linha veio)
- Log de controle (`ingestion_control_log`): 1 linha por arquivo processado, usada para detectar incrementalmente o que já foi ingerido e evitar duplicação

### Objetivo (Card: Ingestão Bronze):
Monitorar a chegada de novos arquivos no Data Lake via loop, ler cada arquivo novo detectado com Spark, adicionar metadados de auditoria (`bronze_ingested_at`, `bronze_source_file`) sem nenhuma transformação, e salvar como Delta na camada Bronze particionado por data de ingestão (ano, mês, dia, hora).

**Pré-requisito:** `00_setup_secrets.ipynb` já executado (secrets já existem no scope `internship-squad1`).


In [0]:
adls_client_id = dbutils.secrets.get(scope="internship-squad1", key="adls-client-id")
adls_tenant_id = dbutils.secrets.get(scope="internship-squad1", key="adls-tenant-id")
adls_client_secret = dbutils.secrets.get(scope="internship-squad1", key="adls-client-secret")
adls_storage_account = dbutils.secrets.get(scope="internship-squad1", key="adls-storage-account")

adls_options = {
    f"fs.azure.account.auth.type.{adls_storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{adls_storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{adls_storage_account}.dfs.core.windows.net": adls_client_id,
    f"fs.azure.account.oauth2.client.secret.{adls_storage_account}.dfs.core.windows.net": adls_client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{adls_storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{adls_tenant_id}/oauth2/token",
}

base_raw = f"abfss://raw@{adls_storage_account}.dfs.core.windows.net/real-time-data"
base_squad = f"abfss://squad1@{adls_storage_account}.dfs.core.windows.net/dupla1"

caminhos = {
    "origem_itens_pedido": f"{base_raw}/*/*/*/*/ecommerce_itens_pedido.parquet",
    "origem_rastreamento": f"{base_raw}/*/*/*/*/ecommerce_rastreamento.parquet",
    "bronze_itens_pedido": f"{base_squad}/bronze/ecommerce_itens_pedido",
    "bronze_rastreamento": f"{base_squad}/bronze/ecommerce_rastreamento_entregas",
    "metadata_control_log": f"{base_squad}/metadata/ingestion_control_log",
}

print("Caminhos configurados no Data Lake:")
for k, v in caminhos.items():
    print(f"  {k}: {v}")


### Etapa 1 — Helper de verificação de tabela Delta

Função usada em vários pontos do notebook pra checar se uma tabela Delta já existe num caminho, sem quebrar se não existir.


In [0]:
def tabela_delta_existe(caminho):
    try:
        spark.read.format("delta").options(**adls_options).load(caminho).limit(1).count()
        return True
    except Exception:
        return False

print("Função de verificação de tabela Delta pronta.")


### Etapa 2 — Detecção incremental: `ecommerce_itens_pedido`

Lê todos os arquivos parquet da origem via Spark nativo, marca a origem de cada linha (`bronze_source_file`), e usa um `left_anti` join contra o log de controle pra filtrar só os arquivos que ainda não foram processados.


In [0]:
from pyspark.sql.functions import current_timestamp, year, month, dayofmonth, hour, col, count, broadcast

def detectar_novos_itens_pedido():
    df_raw = (spark.read
        .options(**adls_options)
        .parquet(caminhos["origem_itens_pedido"])
        .withColumn("bronze_source_file", col("_metadata.file_path")))

    if tabela_delta_existe(caminhos["metadata_control_log"]):
        df_control = spark.read.format("delta").options(**adls_options).load(caminhos["metadata_control_log"])
        df_novos = df_raw.join(
            broadcast(df_control.filter(col("tabela_origem") == "ecommerce_itens_pedido")),
            df_raw.bronze_source_file == df_control.arquivo_processado,
            "left_anti"
        )
    else:
        df_novos = df_raw

    return df_novos

print("Função de detecção para itens_pedido pronta.")


### Etapa 3 — Detecção incremental: `ecommerce_rastreamento` (via SDK)

O parquet de rastreamento tem timestamp em nanossegundos, que `spark.read.parquet` não lê direto — por isso usamos o SDK do Azure (`DataLakeServiceClient`) pra baixar o arquivo e ler via pandas, marcando a origem manualmente (sem `_metadata.file_path`, que só existe na leitura nativa do Spark).


In [0]:
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

credential = ClientSecretCredential(adls_tenant_id, adls_client_id, adls_client_secret)
service_client = DataLakeServiceClient(
    account_url=f"https://{adls_storage_account}.dfs.core.windows.net",
    credential=credential,
)
file_system_client = service_client.get_file_system_client(file_system="raw")

print("Conexão via SDK estabelecida (usada apenas para ecommerce_rastreamento).")


In [0]:
import pandas as pd
from io import BytesIO

def detectar_novos_rastreamento():
    # 1. Lista todos os arquivos de rastreamento disponíveis no Data Lake
    todos_arquivos = [
        p.name for p in file_system_client.get_paths(path="real-time-data", recursive=True)
        if p.name.endswith("ecommerce_rastreamento.parquet")
    ]

    # 2. Descobre quais já foram processados, consultando a tabela de controle
    if tabela_delta_existe(caminhos["metadata_control_log"]):
        df_control = spark.read.format("delta").options(**adls_options).load(caminhos["metadata_control_log"])
        ja_processados = {
            r["arquivo_processado"] for r in
            df_control.filter(col("tabela_origem") == "ecommerce_rastreamento").select("arquivo_processado").collect()
        }
    else:
        ja_processados = set()

    arquivos_novos = [a for a in todos_arquivos if a not in ja_processados]

    if not arquivos_novos:
        return None, []

    # 3. Baixa e lê cada arquivo novo via pandas (contorna o problema de nanosegundos)
    dfs = []
    for caminho in arquivos_novos:
        file_client = file_system_client.get_file_client(caminho)
        conteudo = file_client.download_file().readall()
        pdf = pd.read_parquet(BytesIO(conteudo))
        pdf["bronze_source_file"] = caminho  # marca a origem manualmente (sem _metadata.file_path aqui)
        dfs.append(pdf)

    pdf_final = pd.concat(dfs, ignore_index=True)
    df_novos = spark.createDataFrame(pdf_final)

    return df_novos, arquivos_novos

print("Função de detecção para rastreamento pronta.")


### Etapa 4 — Gravação na Bronze + log de controle

Adiciona os metadados de auditoria (`bronze_ingested_at` e partições de data), grava em modo `append` particionado, e registra cada arquivo processado no log de controle (`ingestion_control_log`) — esse log é o que permite a detecção incremental acima funcionar.


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType

schema_log = StructType([
    StructField("tabela_origem", StringType(), False),
    StructField("arquivo_processado", StringType(), False),
    StructField("total_linhas", LongType(), False),
    StructField("status", StringType(), False)
])

def processar_bronze(df_novos, nome_tabela, caminho_bronze):
    total = df_novos.count()
    if total == 0:
        print(f"    Nenhum arquivo novo para '{nome_tabela}'.")
        return

    resumo_arquivos = (df_novos
        .groupBy("bronze_source_file")
        .agg(count("*").alias("linhas"))
        .collect())

    print(f"    >>> {len(resumo_arquivos)} arquivo(s) novo(s) detectado(s) para '{nome_tabela}' ({total} linhas)")

    df_bronze = (df_novos
        .withColumn("bronze_ingested_at", current_timestamp())
        .withColumn("ano", year(col("bronze_ingested_at")))
        .withColumn("mes", month(col("bronze_ingested_at")))
        .withColumn("dia", dayofmonth(col("bronze_ingested_at")))
        .withColumn("hora", hour(col("bronze_ingested_at"))))

    (df_bronze.write
        .format("delta")
        .options(**adls_options)
        .mode("append")
        .partitionBy("ano", "mes", "dia", "hora")
        .save(caminho_bronze))

    print(f"    [OK] Bronze de '{nome_tabela}' gravada.")

    try:
        log_data = [(nome_tabela, r["bronze_source_file"], int(r["linhas"]), "SUCCESS") for r in resumo_arquivos]
        df_log = (spark.createDataFrame(log_data, schema=schema_log)
                  .withColumn("timestamp_processamento", current_timestamp()))

        (df_log.write
            .format("delta")
            .options(**adls_options)
            .mode("append")
            .save(caminhos["metadata_control_log"]))

        print(f"    [OK] Log de controle atualizado para '{nome_tabela}'.")
    except Exception as e_log:
        print(f"    [ERRO NO LOG] {type(e_log).__name__}: {e_log}")

print("Função de gravação na Bronze (com diagnóstico) pronta.")


### Etapa 5 — Loop de monitoramento

Verifica a cada 15 segundos (por até 90 segundos, ou até interromper manualmente) se chegou arquivo novo em qualquer uma das duas tabelas, processa o que encontrar, e para automaticamente se acumular 5 falhas consecutivas (proteção contra loop infinito em caso de erro persistente).


In [0]:
import time
from datetime import datetime, timedelta

intervalo_segundos = 15
duracao_maxima_segundos = 90
inicio = datetime.now()
fim_previsto = inicio + timedelta(seconds=duracao_maxima_segundos)
falhas_consecutivas = 0
MAX_FALHAS_CONSECUTIVAS = 5

print(f"Iniciando monitoramento às {inicio.strftime('%H:%M:%S')}.")
print(f"Vai rodar até {fim_previsto.strftime('%H:%M:%S')} (ou até você interromper manualmente).\n")

while datetime.now() < fim_previsto:
    try:
        print(f"--- Verificação às {datetime.now().strftime('%H:%M:%S')} ---")

        # ecommerce_itens_pedido (100% Spark nativo)
        df_novos_itens = detectar_novos_itens_pedido()
        processar_bronze(df_novos_itens, "ecommerce_itens_pedido", caminhos["bronze_itens_pedido"])

        # ecommerce_rastreamento (SDK + pandas, por causa do timestamp em nanosegundos)
        df_novos_rastreamento, _ = detectar_novos_rastreamento()
        if df_novos_rastreamento is not None:
            processar_bronze(df_novos_rastreamento, "ecommerce_rastreamento", caminhos["bronze_rastreamento"])
        else:
            print("    Nenhum arquivo novo para 'ecommerce_rastreamento'.")

        falhas_consecutivas = 0

    except KeyboardInterrupt:
        print("\nMonitoramento interrompido manualmente.")
        break
    except Exception as e:
        falhas_consecutivas += 1
        print(f"    [ERRO] Falha na verificação ({falhas_consecutivas}/{MAX_FALHAS_CONSECUTIVAS}): {e}")
        if falhas_consecutivas >= MAX_FALHAS_CONSECUTIVAS:
            print(f"    {MAX_FALHAS_CONSECUTIVAS} falhas seguidas — interrompendo o loop.")
            break

    time.sleep(intervalo_segundos)

print(f"\nMonitoramento finalizado às {datetime.now().strftime('%H:%M:%S')}.")


### Etapa 6 — Validação final

Confere a contagem final de linhas na Bronze das duas tabelas e mostra o log de controle completo, pra auditoria.


In [0]:
print("=== Validação final da Bronze ===\n")

df_bronze_itens = spark.read.format("delta").options(**adls_options).load(caminhos["bronze_itens_pedido"])
total_itens = df_bronze_itens.count()
print(f"ecommerce_itens_pedido (Bronze): {total_itens} linhas (esperado: 129)")

df_bronze_rastreamento = spark.read.format("delta").options(**adls_options).load(caminhos["bronze_rastreamento"])
total_rastreamento = df_bronze_rastreamento.count()
print(f"ecommerce_rastreamento (Bronze): {total_rastreamento} linhas (esperado: 25)")

print("\n=== Validação do log de controle ===\n")
df_log = spark.read.format("delta").options(**adls_options).load(caminhos["metadata_control_log"])
df_log.select("tabela_origem", "arquivo_processado", "total_linhas", "status", "timestamp_processamento").show(truncate=False)


In [0]:
sem_duplicacao = (total_itens == 129) and (total_rastreamento == 25)

if sem_duplicacao:
    print("Status: OK, sem duplicação.")
else:
    print("Status: ATENÇÃO, contagem diferente do esperado.")
